In [4]:
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.datasets import fetch_20newsgroups
from collections import Counter
from scipy.sparse import dok_matrix, csr_matrix
from scipy.sparse.linalg import svds

In [6]:
# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ABDUL_HADI\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ABDUL_HADI\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ABDUL_HADI\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
# Load the 20 Newsgroups dataset
newsgroups = fetch_20newsgroups(subset='all')
documents = newsgroups.data

In [10]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\b\d+\b', '', text)  # Remove numbers
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation
    words = text.split()
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) > 2]  # Remove short words
    return ' '.join(words)

# Preprocess each document
processed_docs = [preprocess_text(doc) for doc in documents]

In [12]:
# Create vocabulary and document-term matrix as a sparse matrix
def create_vocab(docs, min_freq=5):
    vocab = Counter()
    for doc in docs:
        for word in doc.split():
            vocab[word] += 1
    # Filter out low-frequency words and sort by frequency
    return sorted([word for word, freq in vocab.items() if freq >= min_freq])

In [14]:
# Create vocabulary and sparse document-term matrix
vocab = create_vocab(processed_docs)
vocab_index = {word: i for i, word in enumerate(vocab)}

In [16]:
# Initialize a sparse document-term matrix in DOK format
doc_term_matrix_sparse = dok_matrix((len(processed_docs), len(vocab)), dtype=int)

# Fill in the sparse document-term matrix
for doc_idx, doc in enumerate(processed_docs):
    for word in doc.split():
        if word in vocab_index:
            term_idx = vocab_index[word]
            doc_term_matrix_sparse[doc_idx, term_idx] += 1

# Convert to CSR format for efficient SVD
doc_term_matrix_sparse = doc_term_matrix_sparse.tocsr()

# Convert the document-term matrix to float type
doc_term_matrix_sparse = doc_term_matrix_sparse.astype(float)

In [18]:
# Function to apply SVD
def apply_svd(sparse_matrix, num_components):
    # Perform SVD on the sparse matrix
    U, Sigma, VT = svds(sparse_matrix, k=num_components)
    Sigma = np.diag(Sigma)
    return U, Sigma, VT

In [20]:
# Experiment with different numbers of components
num_components_list = [3, 4, 5]
svd_results = {k: apply_svd(doc_term_matrix_sparse, k) for k in num_components_list}

In [22]:
# Calculate probabilities and apply absolute values to avoid negative probabilities
def calculate_probabilities(U, Sigma, VT):
    # Topic-Word Probability Matrix
    topic_word_matrix = np.abs(VT.T)
    topic_word_prob = topic_word_matrix / topic_word_matrix.sum(axis=0, keepdims=True)

    # Document-Topic Probability Matrix
    document_topic_matrix = np.abs(U @ Sigma)
    document_topic_prob = document_topic_matrix / document_topic_matrix.sum(axis=1, keepdims=True)
    
    return topic_word_prob, document_topic_prob

# Calculate probabilities for each number of components
probabilities = {k: calculate_probabilities(*svd_results[k]) for k in num_components_list}


In [24]:
# Display top words and sample document-topic probabilities for each number of components
for k, (topic_word_prob, document_topic_prob) in probabilities.items():
    print(f"\nNumber of Components (Topics): {k}")
    
    # Display the top words per topic
    for i in range(k):
        top_word_indices = topic_word_prob[:, i].argsort()[::-1][:10]
        top_words = [vocab[idx] for idx in top_word_indices]
        print(f"Top words for Topic {i + 1}: {', '.join(top_words)}")
    
    # Display sample document-topic probabilities
    print("\nSample Document-Topic Probabilities:")
    sample_docs = document_topic_prob[:5]  # Display for first 5 documents
    for doc_idx, doc_probs in enumerate(sample_docs):
        topic_probs = ", ".join([f"Topic {i+1}: {prob:.2f}" for i, prob in enumerate(doc_probs)])
        print(f"Document {doc_idx + 1}: {topic_probs}")


Number of Components (Topics): 3
Top words for Topic 1: jpeg, image, file, do, one, people, would, know, color, said
Top words for Topic 2: edu, file, image, one, jpeg, system, also, use, com, program
Top words for Topic 3: max, g9v, b8f, a86, 1d9, giz, bhj, 75u, 34u, 2di

Sample Document-Topic Probabilities:
Document 1: Topic 1: 0.28, Topic 2: 0.72, Topic 3: 0.01
Document 2: Topic 1: 0.20, Topic 2: 0.79, Topic 3: 0.01
Document 3: Topic 1: 0.52, Topic 2: 0.48, Topic 3: 0.00
Document 4: Topic 1: 0.20, Topic 2: 0.80, Topic 3: 0.00
Document 5: Topic 1: 0.21, Topic 2: 0.78, Topic 3: 0.01

Number of Components (Topics): 4
Top words for Topic 1: g9v, b8f, a86, max, 1d9, 2di, 34u, 6um, bxn, 2tm
Top words for Topic 2: jpeg, image, file, do, one, people, would, know, color, said
Top words for Topic 3: edu, file, image, one, jpeg, system, also, use, com, program
Top words for Topic 4: max, g9v, b8f, a86, 1d9, giz, bhj, 75u, 34u, 2di

Sample Document-Topic Probabilities:
Document 1: Topic 1: 0.0